# MILESTONE 2 – DATA CLEANING & ETL

**Nhiệm vụ:** Làm sạch dữ liệu HMIS, xử lý missing/duplicate và xây dựng pipeline ETL bằng Python.

Nối tiếp thiết kế sơ bộ ở Milestone 1 (`docs/milestone1_phan_tich_thiet_ke.docx`), notebook này thực hiện:

1. **Extract** – nạp 19 bảng dữ liệu thô từ `data/raw/`.
2. **Data Profiling** – khảo sát tổng quan chất lượng dữ liệu (shape, kiểu dữ liệu, missing, duplicate).
3. **Phát hiện vấn đề cụ thể** – toàn vẹn tham chiếu (referential integrity), ràng buộc nghiệp vụ, định dạng dữ liệu.
4. **Transform** – làm sạch từng bảng theo đúng thứ tự phụ thuộc (bảng cha trước bảng con).
5. **Load** – xuất dữ liệu đã làm sạch ra `data/processed/` để phục vụ các milestone tiếp theo (xây dựng Fact/Dimension).


## 1. Extract – Nạp dữ liệu thô

In [1]:
import pandas as pd
import numpy as np
import os
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

RAW_DIR = "../data/raw"
CLEAN_DIR = "../data/processed"
os.makedirs(CLEAN_DIR, exist_ok=True)

TABLE_NAMES = [
    "department", "ward", "bed", "disease", "drug_manufacturer", "drug",
    "drug_inventory", "insurance_provider", "diagnostic_test", "employee",
    "doctor", "patient", "patient_insurance", "admission", "billing",
    "billing_detail", "patient_diagnostic", "prescription", "staff_assignment",
]

raw = {name: pd.read_csv(os.path.join(RAW_DIR, f"{name}.csv"), low_memory=False) for name in TABLE_NAMES}

print(f"Đã nạp {len(raw)} bảng, tổng {sum(len(df) for df in raw.values()):,} dòng.")


Đã nạp 19 bảng, tổng 392,534 dòng.


## 2. Data Profiling – Khảo sát tổng quan chất lượng dữ liệu

In [2]:
def profile_table(name, df):
    pk = df.columns[0]
    return {
        "table": name,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "full_duplicate_rows": int(df.duplicated().sum()),
        "duplicate_pk": int(df[pk].duplicated().sum()),
    }

profiling_df = pd.DataFrame([profile_table(name, df) for name, df in raw.items()])
profiling_df


,table,rows,columns,missing_values,full_duplicate_rows,duplicate_pk
0,department,11,5,0,0,0
1,ward,27,5,0,0,0
2,bed,415,4,0,0,0
3,disease,20,3,0,0,0
4,drug_manufacturer,300,5,0,0,0
5,drug,250,6,0,0,0
6,drug_inventory,250,6,0,0,0
7,insurance_provider,50,5,0,0,0
8,diagnostic_test,9,5,0,0,0
9,employee,500,7,0,0,0


**Nhận xét:** Ở mức bề mặt, bộ dữ liệu HMIS không có giá trị NULL hay dòng trùng lặp hoàn toàn — đây là
dữ liệu tổng hợp (synthetic) khá sạch. Tuy nhiên "sạch bề mặt" không có nghĩa là sẵn sàng để dùng: cần kiểm tra
thêm toàn vẹn tham chiếu và các ràng buộc nghiệp vụ ở bước 3 bên dưới.

## 3. Phát hiện vấn đề chất lượng dữ liệu cụ thể

### 3.1 Toàn vẹn tham chiếu (Referential Integrity)

In [3]:
# (bảng con, cột FK, bảng cha, cột PK cha)
FK_LIST = [
    ("ward", "department_id", "department", "department_id"),
    ("bed", "ward_id", "ward", "ward_id"),
    ("drug", "manufacturer_id", "drug_manufacturer", "manufacturer_id"),
    ("drug_inventory", "drug_id", "drug", "drug_id"),
    ("diagnostic_test", "department_id", "department", "department_id"),
    ("employee", "department_id", "department", "department_id"),
    ("doctor", "employee_id", "employee", "employee_id"),
    ("patient_insurance", "patient_id", "patient", "patient_id"),
    ("patient_insurance", "insurance_provider_id", "insurance_provider", "insurance_provider_id"),
    ("admission", "patient_id", "patient", "patient_id"),
    ("admission", "department_id", "department", "department_id"),
    ("admission", "ward_id", "ward", "ward_id"),
    ("admission", "bed_id", "bed", "bed_id"),
    ("admission", "disease_id", "disease", "disease_id"),
    ("billing", "admission_id", "admission", "admission_id"),
    ("billing_detail", "bill_id", "billing", "bill_id"),
    ("patient_diagnostic", "admission_id", "admission", "admission_id"),
    ("patient_diagnostic", "test_id", "diagnostic_test", "test_id"),
    ("patient_diagnostic", "doctor_id", "doctor", "doctor_id"),
    ("prescription", "admission_id", "admission", "admission_id"),
    ("prescription", "drug_id", "drug", "drug_id"),
    ("staff_assignment", "employee_id", "employee", "employee_id"),
    ("staff_assignment", "ward_id", "ward", "ward_id"),
]

fk_report = []
for child, fk_col, parent, pk_col in FK_LIST:
    orphan_count = (~raw[child][fk_col].isin(raw[parent][pk_col])).sum()
    fk_report.append({"child_table": child, "fk_column": fk_col, "parent_table": parent, "orphan_rows": int(orphan_count)})

pd.DataFrame(fk_report)


,child_table,fk_column,parent_table,orphan_rows
0,ward,department_id,department,0
1,bed,ward_id,ward,0
2,drug,manufacturer_id,drug_manufacturer,0
3,drug_inventory,drug_id,drug,0
4,diagnostic_test,department_id,department,0
5,employee,department_id,department,0
6,doctor,employee_id,employee,0
7,patient_insurance,patient_id,patient,0
8,patient_insurance,insurance_provider_id,insurance_provider,0
9,admission,patient_id,patient,0


Không có dòng nào bị "mồ côi" (orphan) tham chiếu đến khóa cha không tồn tại — toàn vẹn tham chiếu được đảm bảo.

### 3.2 Ràng buộc nghiệp vụ (Business Rules)

In [4]:
admission_tmp = raw["admission"].copy()
admission_tmp["admission_date"] = pd.to_datetime(admission_tmp["admission_date"])
admission_tmp["discharge_date"] = pd.to_datetime(admission_tmp["discharge_date"])
admission_tmp["length_of_stay_days"] = (admission_tmp["discharge_date"] - admission_tmp["admission_date"]).dt.days

print("Số lượt nhập viện có discharge_date < admission_date (LOS âm):",
      (admission_tmp["length_of_stay_days"] < 0).sum())

# Trùng khoảng thời gian nằm viện của cùng một bệnh nhân
overlap_flags = pd.Series(False, index=admission_tmp.index)
for _, grp in admission_tmp.sort_values("admission_date").groupby("patient_id"):
    prev_end = None
    for idx, row in grp.iterrows():
        if prev_end is not None and row["admission_date"] < prev_end:
            overlap_flags.loc[idx] = True
        if pd.notna(row["discharge_date"]):
            prev_end = row["discharge_date"] if prev_end is None else max(prev_end, row["discharge_date"])

print("Số lượt nhập viện trùng khoảng thời gian với lượt khác của cùng bệnh nhân:", int(overlap_flags.sum()))
admission_tmp[overlap_flags].sort_values("patient_id")[
    ["admission_id", "patient_id", "admission_date", "discharge_date"]
].head(6)


Số lượt nhập viện có discharge_date < admission_date (LOS âm): 0


Số lượt nhập viện trùng khoảng thời gian với lượt khác của cùng bệnh nhân: 135


,admission_id,patient_id,admission_date,discharge_date
18613,18614,10,2020-11-12,2020-11-17
29496,29497,104,2020-03-05,2020-03-11
29654,29655,119,2020-11-18,2020-11-28
19629,19630,674,2022-09-13,2022-09-14
39762,39763,685,2023-05-10,2023-05-11
11498,11499,795,2021-10-30,2021-11-01


**Vấn đề phát hiện:** 135 lượt nhập viện trùng khoảng thời gian với lượt nhập viện khác của cùng bệnh nhân
(về mặt nghiệp vụ, một bệnh nhân không thể nằm viện ở hai nơi cùng lúc). Không đủ căn cứ để xác định bản ghi nào
sai, nên các dòng này sẽ được **giữ lại và gắn cờ** `overlapping_stay_flag` ở bước làm sạch, thay vì xóa —
tương tự tinh thần cảnh báo (DSS) đã đề ra ở Milestone 1.

In [5]:
billing_tmp = raw["billing"]
mismatch = (billing_tmp["insurance_covered_amount"] + billing_tmp["patient_payable_amount"] - billing_tmp["total_amount"]).abs() > 0.01
print("Số hóa đơn có insurance_covered + patient_payable != total_amount:", int(mismatch.sum()))

pi_tmp = raw["patient_insurance"]
print("Số hợp đồng bảo hiểm có coverage_percentage ngoài [0, 100]:",
      int(((pi_tmp["coverage_percentage"] < 0) | (pi_tmp["coverage_percentage"] > 100)).sum()))

inv_tmp = raw["drug_inventory"]
low_stock = inv_tmp["current_stock"] < inv_tmp["reorder_level"]
print("Số loại thuốc có current_stock < reorder_level (cảnh báo tồn kho thấp):", int(low_stock.sum()),
      "| khớp với inventory_status='Low':", int((inv_tmp.loc[low_stock, "inventory_status"] == "Low").all()))


Số hóa đơn có insurance_covered + patient_payable != total_amount: 0
Số hợp đồng bảo hiểm có coverage_percentage ngoài [0, 100]: 0
Số loại thuốc có current_stock < reorder_level (cảnh báo tồn kho thấp): 44 | khớp với inventory_status='Low': 1


Các hóa đơn khớp tổng tiền, tỉ lệ bảo hiểm chi trả nằm trong khoảng hợp lệ, và cờ tồn kho thấp khớp đúng với
`inventory_status`. Đây là các bước kiểm tra **phòng vệ** (defensive) — hiện chưa phát hiện vi phạm, nhưng
pipeline vẫn cần giữ lại các luật này để tự động phát hiện nếu dữ liệu batch sau có vấn đề.

### 3.3 Định dạng số điện thoại không nhất quán (`patient.contact_number`)

In [6]:
sample = raw["patient"]["contact_number"].sample(8, random_state=1).tolist()
sample


['001-877-802-6503x90630',
 '+1-920-565-6284x53233',
 '751.905.6125',
 '(965)250-2454x582',
 '(635)664-2827',
 '6816374924',
 '+1-476-589-6556x58057',
 '504.738.4258x405']

Cột `contact_number` trộn lẫn nhiều định dạng: `+1-xxx-xxx-xxxx`, `(xxx)xxx-xxxx`, `xxx.xxx.xxxx`,
`001-xxx-xxx-xxxx`, kèm phần mở rộng `x#####` không cố định. Cần chuẩn hóa về một định dạng thống nhất
(số 10 chữ số + phần mở rộng riêng) để có thể so khớp/loc dữ liệu về sau.

### 3.4 Giá trị NULL hợp lệ theo thiết kế (`billing_detail.reference_id`)

In [7]:
raw["billing_detail"].groupby("charge_type")["reference_id"].apply(lambda s: s.isna().sum()).rename("null_count")


charge_type
Drug         31344
Procedure    13575
Room             0
Test         22483
Name: null_count, dtype: int64

`reference_id` là khóa tham chiếu đa hình (trỏ đến `bed_id` khi `charge_type = Room`, đến `test_id` khi
`charge_type = Test`...). NULL ở `charge_type = Drug/Procedure` là **hợp lệ theo thiết kế** vì khoản thu đó
không gắn với một bản ghi tham chiếu cụ thể — không phải lỗi dữ liệu, nên sẽ không bị xử lý như missing value.

## 4. Transform – Làm sạch dữ liệu

### 4.1 Cấu hình bảng: thứ tự phụ thuộc, khóa chính, khóa ngoại, cột cần chuẩn hóa

In [8]:
# Danh sách bảng theo đúng thứ tự phụ thuộc: bảng cha luôn đứng trước bảng con,
# để khi kiểm tra FK của bảng con thì khóa chính của bảng cha đã được làm sạch xong.
TABLE_CONFIG = [
    dict(name="department", pk="department_id",
         text_cols=["department_name", "department_type", "status"],
         numeric_rules=[("floor_number", "ge", 0)]),
    dict(name="ward", pk="ward_id", fk=[("department_id", "department")],
         text_cols=["ward_name", "ward_type"], numeric_rules=[("total_beds", "gt", 0)]),
    dict(name="bed", pk="bed_id", fk=[("ward_id", "ward")],
         text_cols=["bed_number", "bed_status"]),
    dict(name="disease", pk="disease_id", text_cols=["disease_name", "disease_category"]),
    dict(name="drug_manufacturer", pk="manufacturer_id",
         text_cols=["manufacturer_name", "country", "contract_status"],
         numeric_rules=[("reliability_rating", "ge", 0), ("reliability_rating", "le", 5)]),
    dict(name="drug", pk="drug_id", fk=[("manufacturer_id", "drug_manufacturer")],
         text_cols=["drug_name", "brand_name", "drug_category"], numeric_rules=[("unit_cost", "gt", 0)]),
    dict(name="drug_inventory", pk="inventory_id", fk=[("drug_id", "drug")],
         date_cols=["last_restock_date"], text_cols=["inventory_status"],
         numeric_rules=[("current_stock", "ge", 0), ("reorder_level", "ge", 0)]),
    dict(name="insurance_provider", pk="insurance_provider_id",
         text_cols=["provider_name", "provider_type"], numeric_rules=[("coverage_limit", "gt", 0)]),
    dict(name="diagnostic_test", pk="test_id", fk=[("department_id", "department")],
         text_cols=["test_name", "test_category"], numeric_rules=[("standard_cost", "gt", 0)]),
    dict(name="employee", pk="employee_id", fk=[("department_id", "department")],
         date_cols=["date_of_joining"], text_cols=["employee_name", "gender", "role", "employment_type"]),
    dict(name="doctor", pk="doctor_id", fk=[("employee_id", "employee")],
         text_cols=["specialization", "qualification"], numeric_rules=[("experience_years", "ge", 0)]),
    dict(name="patient", pk="patient_id", date_cols=["date_of_birth"],
         text_cols=["gender", "blood_group", "city"], special="patient"),
    dict(name="patient_insurance", pk="patient_insurance_id",
         fk=[("patient_id", "patient"), ("insurance_provider_id", "insurance_provider")],
         date_cols=["policy_start_date", "policy_end_date"],
         numeric_rules=[("coverage_percentage", "ge", 0), ("coverage_percentage", "le", 100)]),
    dict(name="admission", pk="admission_id",
         fk=[("patient_id", "patient"), ("department_id", "department"), ("ward_id", "ward"),
             ("bed_id", "bed"), ("disease_id", "disease")],
         date_cols=["admission_date", "discharge_date"],
         text_cols=["admission_type", "admission_status"], special="admission"),
    dict(name="billing", pk="bill_id", fk=[("admission_id", "admission")],
         date_cols=["bill_date"], text_cols=["payment_status", "payment_mode"],
         numeric_rules=[("total_amount", "gt", 0)], special="billing"),
    dict(name="billing_detail", pk="billing_detail_id", fk=[("bill_id", "billing")],
         text_cols=["charge_type"], numeric_rules=[("amount", "gt", 0)], special="billing_detail"),
    dict(name="patient_diagnostic", pk="patient_diagnostic_id",
         fk=[("admission_id", "admission"), ("test_id", "diagnostic_test"), ("doctor_id", "doctor")],
         date_cols=["test_date"], text_cols=["result_status"]),
    dict(name="prescription", pk="prescription_id", fk=[("admission_id", "admission"), ("drug_id", "drug")],
         text_cols=["dosage", "frequency"], numeric_rules=[("duration_days", "gt", 0)]),
    dict(name="staff_assignment", pk="assignment_id",
         fk=[("employee_id", "employee"), ("ward_id", "ward")], text_cols=["shift"]),
]
TABLE_PK = {c["name"]: c["pk"] for c in TABLE_CONFIG}
print(f"Đã cấu hình {len(TABLE_CONFIG)} bảng theo thứ tự phụ thuộc.")


Đã cấu hình 19 bảng theo thứ tự phụ thuộc.


### 4.2 Các hàm làm sạch dùng chung

In [9]:
NUMERIC_OPS = {"gt": lambda s, v: s > v, "ge": lambda s, v: s >= v,
               "lt": lambda s, v: s < v, "le": lambda s, v: s <= v}


def strip_text_columns(df, cols):
    for col in cols:
        df[col] = df[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
    return df


def parse_date_columns(df, cols):
    for col in cols:
        df[col] = pd.to_datetime(df[col], errors="coerce")
    return df


def drop_duplicates(df, pk):
    df = df.drop_duplicates(keep="first")
    df = df.drop_duplicates(subset=[pk], keep="first")
    return df


def drop_missing_required(df, required_cols):
    cols = [c for c in required_cols if c in df.columns]
    return df.dropna(subset=cols) if cols else df


def drop_fk_orphans(df, fk_list, cleaned):
    for col, parent in fk_list:
        valid_keys = set(cleaned[parent][TABLE_PK[parent]])
        df = df[df[col].isin(valid_keys)]
    return df


def apply_numeric_rules(df, rules):
    for col, op, value in rules:
        df = df[NUMERIC_OPS[op](df[col], value)]
    return df


_PHONE_EXT_RE = re.compile(r"(?:ext\.?|x)\s*(\d+)$", re.IGNORECASE)


def clean_patient_contact(df):
    raw_numbers = df["contact_number"].astype(str)
    ext = raw_numbers.str.extract(_PHONE_EXT_RE)[0]
    without_ext = raw_numbers.str.replace(_PHONE_EXT_RE, "", regex=True)
    digits = without_ext.str.replace(r"\D", "", regex=True).str[-10:]
    df["contact_number"] = digits
    df["contact_extension"] = ext.fillna("")
    return df


def clean_admission_rules(df):
    df = df.sort_values(["patient_id", "admission_date"]).reset_index(drop=True)
    df["length_of_stay_days"] = (df["discharge_date"] - df["admission_date"]).dt.days
    df = df[df["length_of_stay_days"] >= 0]

    overlap = pd.Series(False, index=df.index)
    for _, grp in df.groupby("patient_id"):
        prev_end = None
        for idx, row in grp.sort_values("admission_date").iterrows():
            if prev_end is not None and row["admission_date"] < prev_end:
                overlap.loc[idx] = True
            if pd.notna(row["discharge_date"]):
                prev_end = row["discharge_date"] if prev_end is None else max(prev_end, row["discharge_date"])
    df["overlapping_stay_flag"] = overlap
    return df


def clean_billing_rules(df):
    consistent = (df["insurance_covered_amount"] + df["patient_payable_amount"] - df["total_amount"]).abs() <= 0.01
    return df[consistent]


def clean_billing_detail_rules(df):
    df["reference_id"] = df["reference_id"].astype("Int64")
    return df


### 4.3 Áp dụng pipeline làm sạch cho từng bảng (theo đúng thứ tự phụ thuộc)

In [10]:
cleaned = {}
cleaning_log = []

for cfg in TABLE_CONFIG:
    name = cfg["name"]
    df = raw[name].copy()
    rows_before = len(df)

    df = strip_text_columns(df, cfg.get("text_cols", []))
    df = parse_date_columns(df, cfg.get("date_cols", []))
    df = drop_duplicates(df, cfg["pk"])

    required = [cfg["pk"]] + [c for c, _ in cfg.get("fk", [])] + cfg.get("date_cols", [])
    df = drop_missing_required(df, required)

    if cfg.get("fk"):
        df = drop_fk_orphans(df, cfg["fk"], cleaned)
    if cfg.get("numeric_rules"):
        df = apply_numeric_rules(df, cfg["numeric_rules"])

    special = cfg.get("special")
    if special == "patient":
        df = clean_patient_contact(df)
    elif special == "admission":
        df = clean_admission_rules(df)
    elif special == "billing":
        df = clean_billing_rules(df)
    elif special == "billing_detail":
        df = clean_billing_detail_rules(df)

    cleaned[name] = df
    cleaning_log.append({"table": name, "rows_before": rows_before, "rows_after": len(df),
                          "rows_removed": rows_before - len(df)})

cleaning_summary = pd.DataFrame(cleaning_log)
cleaning_summary


,table,rows_before,rows_after,rows_removed
0,department,11,11,0
1,ward,27,27,0
2,bed,415,415,0
3,disease,20,20,0
4,drug_manufacturer,300,300,0
5,drug,250,250,0
6,drug_inventory,250,250,0
7,insurance_provider,50,50,0
8,diagnostic_test,9,9,0
9,employee,500,500,0


**Nhận xét:** Vì dữ liệu gốc vốn đã sạch (không missing/duplicate/FK lỗi), số dòng bị loại ở hầu hết các
bảng là 0 — các bước dọn dẹp chung đóng vai trò **phòng vệ**, tự động xử lý nếu dữ liệu batch sau có vấn đề.
Các thay đổi thực sự đã áp dụng: chuẩn hóa `contact_number`/`contact_extension` (patient), tính
`length_of_stay_days` và gắn cờ `overlapping_stay_flag` (admission), ép kiểu `reference_id` về Int64
(billing_detail).

In [11]:
cleaned["patient"][["patient_id", "contact_number", "contact_extension"]].head()


,patient_id,contact_number,contact_extension
0,1,7923420981,
1,2,7937250800,
2,3,3306173749,232
3,4,4266116235,07684
4,5,2153302831,0821


In [12]:
cleaned["admission"][cleaned["admission"]["overlapping_stay_flag"]][
    ["admission_id", "patient_id", "admission_date", "discharge_date", "length_of_stay_days"]
].head()


,admission_id,patient_id,admission_date,discharge_date,length_of_stay_days
14,18614,10,2020-11-12,2020-11-17,5
158,29497,104,2020-03-05,2020-03-11,6
180,34392,119,2020-11-18,2020-11-20,2
1031,19630,674,2022-09-13,2022-09-14,1
1049,39763,685,2023-05-10,2023-05-11,1


## 5. Load – Xuất dữ liệu đã làm sạch

In [13]:
for name, df in cleaned.items():
    df.to_csv(os.path.join(CLEAN_DIR, f"{name}.csv"), index=False)

print(f"Đã ghi {len(cleaned)} bảng đã làm sạch vào thư mục '{CLEAN_DIR}/'.")


Đã ghi 19 bảng đã làm sạch vào thư mục 'data/processed/'.


## 6. Tổng kết

- Toàn vẹn tham chiếu (referential integrity) giữa 19 bảng được xác nhận đầy đủ, không có FK "mồ côi".
- Không có missing value hay dòng trùng lặp hoàn toàn ở dữ liệu gốc.
- Đã phát hiện và xử lý 3 vấn đề chất lượng dữ liệu thực tế:
  1. Định dạng `contact_number` không nhất quán → chuẩn hóa về 10 chữ số + `contact_extension` riêng.
  2. 135 lượt nhập viện trùng khoảng thời gian của cùng bệnh nhân → giữ lại, gắn cờ `overlapping_stay_flag`
     để nhà quản lý xem xét (không đủ căn cứ để tự động xóa).
  3. `billing_detail.reference_id` NULL ở charge_type Drug/Procedure là hợp lệ theo thiết kế (khóa tham chiếu
     đa hình), không phải lỗi.
- Dữ liệu đã làm sạch tại `data/processed/` sẵn sàng để xây dựng các bảng Fact/Dimension theo thiết kế
  sơ bộ ở Milestone 1.
